In this notebook, we load models trained with best hyperparameters and analyze their performance.

In [ ]:
import json
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

import keras
import tensorflow as tf

from load_dataset import build_dataset, make_tf_dataset, make_torch_dataset
from model_swin_unetr import build_swin_unetr_mc

import metric_tf
import metric_torch

print(f"Tensorflow devices: {tf.config.list_physical_devices()}")

# Load trained models

In [ ]:
unet_model_path = "output_unet3d_all.keras"
swin_model_path = "output_swin_unetr_all.pth"

unet_model = keras.models.load_model(unet_model_path, compile = False)
unet_model.compile(
    optimizer = keras.optimizers.Adam(),
    loss = metric_tf.bce_dice_loss,
    metrics=["accuracy"]
)

swin_model = build_swin_unetr_mc()
swin_model_state_dict = torch.load(swin_model_path, weights_only = False, map_location = device)
swin_model.load_state_dict(swin_model_state_dict["model_state_dict"])
swin_model = swin_model.to(device)

# Load test dataset

In [ ]:
x_test, y_test = build_dataset(dataset="test")

# U-Net Inference

In [ ]:
print(f"Running U-Net inference")
test_tf_dataset = make_tf_dataset(x_test, y_test, training = False)
unet_predictions = unet_model.predict(test_tf_dataset)
print(f"Predictions shape: {unet_predictions.shape}")

# U-Net Metrics

In [ ]:
import numpy as np

y_true_tf = tf.constant(y_test, dtype = tf.float32)
y_prediction_tf = tf.constant(unet_predictions, dtype = tf.float32)

unet_loss = metric_tf.bce_dice_loss(y_true_tf, y_prediction_tf).numpy()
unet_dice = metric_tf.dice_coef(y_true_tf, y_prediction_tf).numpy()
unet_hausdorffdistance = metric_torch.hausdorff_distance( # Squeeze below to remove the last dimension
    torch.from_numpy(y_test).squeeze(-1),
    torch.from_numpy(unet_predictions).squeeze(-1)
)

print(f"U-Net Loss: {unet_loss}")
print(f"U-Net Dice: {unet_dice}")
print(f"U-Net Hausdorff Distance: {unet_hausdorffdistance}")

# Swin-transformer Inference

In [ ]:
test_torch_dataset = make_torch_dataset(x_test, y_test, training = False)

swin_model.eval()
all_predictions = []
all_labels = []

with torch.inference_mode():
    for batch_x, batch_y in test_torch_dataset:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        prediction = swin_model(batch_x)
        prediction = torch.sigmoid(prediction)
        all_predictions.append(prediction)
        all_labels.append(batch_y)

# Join sequence of tensors
all_predictions = torch.cat(all_predictions, dim = 0)
all_labels = torch.cat(all_labels, dim = 0)

# Swin-transformer Metrics

In [ ]:
swin_loss = metric_torch.bce_dice_loss(all_labels, all_predictions).item()
swin_dice = metric_torch.dice_coef(all_labels, all_predictions).item()
swin_hausdorffdistance = metric_torch.hausdorff_distance(all_labels, all_predictions)

print(f"Swin Loss: {swin_loss}")
print(f"Swin Dice: {swin_dice}")
print(f"Swin Hausdorff Distance: {swin_hausdorffdistance}")

# Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open("unet_history_all.json") as unet_file:
    unet_history = json.load(unet_file)
with open("swin_history_all.json") as swin_file:
    swin_history = json.load(swin_file)

epochs = range(1, len(unet_history["loss"]) + 1)

# U-Net Loss plot
plt.plot(epochs, unet_history["loss"], label = "Train")
plt.plot(epochs, unet_history["val_loss"], label = "Validation")
plt.title("U-Net Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("plots/training_curves/unet_loss.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# U-Net Dice plot
plt.plot(epochs, unet_history["dice_coef"], label = "Train")
plt.plot(epochs, unet_history["val_dice_coef"], label = "Validation")
plt.title("U-Net Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Coef")
plt.legend()
plt.savefig("plots/training_curves/unet_dice.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Swin Loss plot
plt.plot(epochs, swin_history["train_loss"], label = "Train")
plt.plot(epochs, swin_history["val_loss"], label = "Validation")
plt.title("Swin-UNETR Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("plots/training_curves/swin_loss.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Swin Dice plot
plt.plot(epochs, swin_history["train_dice"], label = "Train")
plt.plot(epochs, swin_history["val_dice"], label = "Validation")
plt.title("Swin-UNETR Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Coef")
plt.legend()
plt.savefig("plots/training_curves/swin_dice.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Plot Test Metrics Bar Chart

In [ ]:
models = ["U-Net", "Swin-UNETR"]
colors = ["darkblue", "darkorange"]

# Loss comparisons
losses = [unet_loss, swin_loss]
plt.bar(models, losses, color = colors)
plt.title("Binary Cross-Entropy + Dice Loss Comparison") # Lower is better
plt.ylim(0, 1)
plt.savefig("plots/metrics/bce_dice_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Dice Coefficients
coefficients = [unet_dice, swin_dice]
plt.bar(models, coefficients, color = colors)
plt.title("Dice Coefficient Comparison") # Higher is better
plt.ylim(0, 1)
plt.savefig("plots/metrics/dice_coef_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Hausdorff Distances
hausdorffdistances = [unet_hausdorffdistance, swin_hausdorffdistance]
plt.bar(models, hausdorffdistances, color = colors)
plt.title("Hausdorff Distance Comparison") # Higher is better
plt.ylim(0, 30)
plt.savefig("plots/metrics/hausdorffdistances_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Plot Sample Predictions

In [ ]:
# Choose which samples and get the middle slice
sample_indices = [0, 1, 2]
mid_slice = x_test.shape[1] // 2

swin_predictions_np = all_predictions.cpu().numpy() # Currently looking like [N, 1, D, H, W]
swin_predictions_np = swin_predictions_np[:, 0, :, :, :] # [N, D, H, W] so it squeezes the channel dim

fig, axes = plt.subplots(3, 4, figsize = (16, 12))
fig.suptitle("Middle Slice Sample Predictions")
col_titles = ["Input (T1)", "Ground Truth", "U-Net Prediction", "Swin-UNETR Prediction"]

for row, idx in enumerate(sample_indices):
    # T1 slice
    input_slice = x_test[idx, mid_slice, :, :, 0]
    # Binary ground truth
    ground_truth_slice = y_test[idx, mid_slice, :, :, 0]
    # U-Net predicted probability map
    unet_slice = unet_predictions[idx, mid_slice, :, :, 0]
    # Swin predicted probability map
    swin_slice = swin_predictions_np[idx, mid_slice, :, :]

    axes[row, 0].imshow(input_slice, cmap="gray")
    axes[row, 1].imshow(ground_truth_slice, cmap="gray")
    axes[row, 2].imshow(unet_slice, cmap="gray")
    axes[row, 3].imshow(swin_slice, cmap="gray")

    for col in range(4):
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(col_titles[col])

plt.tight_layout()
plt.savefig("plots/samples/sample_predictions.png", dpi = 1200, bbox_inches = 'tight')
plt.show()